# Dynamic-Mode Visualization

Thin plotting workflow for standardized ChronoPack gene-trajectory clustering outputs. Reusable plotting logic lives in `chronopack.pl`; this notebook keeps only dataset-specific selections and output paths.

In [ ]:
from pathlib import Path

import anndata as ad
import chronopack as cp

workspace_notebook_dir = Path.cwd().resolve()
OUTPUT_DIR = Path(
    __import__("os").environ.get(
        "CHRONOPACK_DIMRED_OUTPUT_DIR",
        workspace_notebook_dir / "self_contained_outputs/global_analysis_extended",
    )
).expanduser().resolve()
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

adata_path = OUTPUT_DIR / "gene_trajectory_clusters.h5ad"
if not adata_path.exists():
    raise FileNotFoundError(
        f"Missing {adata_path}. Run DimensionalityReduction&Clustering.self_contained.ipynb first "
        "or set CHRONOPACK_DIMRED_OUTPUT_DIR to a directory containing gene_trajectory_clusters.h5ad."
    )
adata = ad.read_h5ad(adata_path)
fit_key = "gene_trajectories"
cluster_key = "gene_trajectory_clusters"

print(adata)
print(adata.var[f"{cluster_key}_broad_cluster"].value_counts())

## Broad Dynamic Modes

In [ ]:
broad_order = [
    "trend-up",
    "impulse-up_1",
    "impulse-up_2",
    "trend-down_1",
    "trend-down_2",
    "transient-down",
]

broad = cp.pl.cluster_trajectories(
    adata,
    groupby="broad_cluster",
    fit_key=fit_key,
    clustering_key=cluster_key,
    normalize=True,
    output=PLOT_DIR / "broad_dynamic_modes.svg",
    title="Broad dynamic modes: median +/- IQR",
)
broad.output

## Representative Genes

In [ ]:
def top_rows_by_broad_cluster(adata, broad_clusters, n=3):
    rows = []
    broad_col = f"{cluster_key}_broad_cluster"
    wald_col = f"{fit_key}_waldStat_1"
    for mode in broad_clusters:
        sub = adata.var.loc[adata.var[broad_col].astype(str) == mode, [wald_col]].copy()
        sub["row_id"] = sub.index.astype(str)
        rows.extend(sub.sort_values([wald_col, "row_id"], ascending=[False, True]).head(n).index.tolist())
    return rows

broad_examples = top_rows_by_broad_cluster(adata, broad_order, n=3)
examples = cp.pl.gene_trajectory(
    adata,
    broad_examples,
    fit_key=fit_key,
    clustering_key=cluster_key,
    normalize=False,
    output=PLOT_DIR / "broad_cluster_example_genes.svg",
    width=12,
    height=12,
    title="Top example genes from each broad cluster",
)
examples.output

## Fine Clusters Within Broad Modes

In [ ]:
def fine_column_for_broad(mode):
    if mode in {"trend-up", "impulse-up_1", "impulse-up_2"}:
        return "cluster_fine_induced"
    return "cluster_fine_repressed"

fine_outputs = {}
for mode in broad_order:
    fine_col = fine_column_for_broad(mode)
    broad_col = f"{cluster_key}_broad_cluster"
    full_fine_col = f"{cluster_key}_{fine_col}"
    rows = adata.var_names[
        (adata.var[broad_col].astype(str) == mode)
        & (adata.var[full_fine_col].astype(str) != "-1")
    ].tolist()
    if rows:
        res = cp.pl.cluster_trajectories(
            adata,
            rows=rows,
            groupby=fine_col,
            fit_key=fit_key,
            clustering_key=cluster_key,
            normalize=True,
            output=PLOT_DIR / f"fine_clusters.{mode}.svg",
            title=f"{mode}: fine-cluster median curves",
        )
        fine_outputs[mode] = res.output
fine_outputs

## Trajectory Feature Space

In [ ]:
features = cp.pl.trajectory_features(
    adata,
    x="PC1",
    y="PC2",
    color="broad_cluster",
    fit_key=fit_key,
    clustering_key=cluster_key,
    output=PLOT_DIR / "trajectory_feature_space.svg",
    title="Gene trajectory feature space",
)
features.output

## Fine-Cluster Representative Table

In [ ]:
fine_examples = []
for mode in broad_order:
    fine_col = fine_column_for_broad(mode)
    broad_col = f"{cluster_key}_broad_cluster"
    full_fine_col = f"{cluster_key}_{fine_col}"
    rows_bc = adata.var.index[adata.var[broad_col].astype(str) == mode]
    for fine in sorted(set(adata.var.loc[rows_bc, full_fine_col].astype(str)) - {"-1"}):
        sub = adata.var.loc[
            rows_bc[adata.var.loc[rows_bc, full_fine_col].astype(str) == fine],
            [f"{fit_key}_waldStat_1"],
        ].copy()
        sub["row_id"] = sub.index.astype(str)
        fine_examples.extend(sub.sort_values([f"{fit_key}_waldStat_1", "row_id"], ascending=[False, True]).head(1).index.tolist())

fine_example_plot = cp.pl.gene_trajectory(
    adata,
    fine_examples,
    fit_key=fit_key,
    clustering_key=cluster_key,
    normalize=False,
    output=PLOT_DIR / "fine_cluster_example_genes.svg",
    width=14,
    height=14,
    title="Top representative gene from each fine cluster",
)
fine_example_plot.output